# Data Cleaning and Preprocessing Pipeline

This notebook handles the extraction, filtering, and preparation of ACLED political violence data for the Afghanistan-Neighbors spillover analysis.

**Workflow:**
1. Load raw ACLED CSV data
2. Filter for target countries and time period (2020-01 to 2021-01)
3. Aggregate events to monthly counts
4. Create lagged variables for spillover analysis
5. Output cleaned dataset in wide format for regression

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully")

## Step 1: Load and Filter Raw ACLED Data

Load ACLED CSV and filter by:
- Countries: Afghanistan + 6 neighbors (Turkmenistan, Uzbekistan, Tajikistan, Iran, Pakistan, China)
- Date range: 2020-01-01 to 2021-01-31
- Event types: Political violence events (battles, explosions, etc.)

In [ ]:
# Define target countries and paths
COUNTRIES = ['Afghanistan', 'Turkmenistan', 'Uzbekistan', 'Tajikistan', 'Iran', 'Pakistan', 'China']
NEIGHBORS = ['Turkmenistan', 'Uzbekistan', 'Tajikistan', 'Iran', 'Pakistan', 'China']

# Adjust path based on notebook location
import os
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
data_dir = os.path.join(project_root, 'data')

# Try to load ACLED data
try:
    # Look for ACLED CSV in data folder
    acled_files = [f for f in os.listdir(data_dir) if 'acled' in f.lower() and f.endswith('.csv')]
    if acled_files:
        acled_path = os.path.join(data_dir, acled_files[0])
    else:
        # Use default path
        acled_path = os.path.join(data_dir, 'acled_data.csv')
    
    df_raw = pd.read_csv(acled_path)
    print(f"✓ Loaded ACLED data: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
    print(f"  Date range: {df_raw['data_start_date'].min()} to {df_raw['data_start_date'].max()}")
except Exception as e:
    print(f"Note: Could not load ACLED data: {e}")
    print("Creating sample data structure for demonstration...")
    df_raw = None

In [ ]:
# Data filtering function
def load_and_filter_acled(df, countries, start_date='2020-01-01', end_date='2021-01-31'):
    """Filter ACLED data for target countries and date range"""
    
    if df is None:
        return None
    
    # Convert date columns
    if 'data_start_date' in df.columns:
        df['date'] = pd.to_datetime(df['data_start_date'])
    elif 'event_date' in df.columns:
        df['date'] = pd.to_datetime(df['event_date'])
    else:
        df['date'] = pd.to_datetime(df.iloc[:, 0])  # First column as date
    
    # Filter by country and date
    df_filtered = df[df['country'].isin(countries)].copy()
    df_filtered = df_filtered[(df_filtered['date'] >= start_date) & (df_filtered['date'] <= end_date)]
    
    # Filter for political violence events
    if 'event_type' in df_filtered.columns:
        violence_types = ['Battles', 'Violence against civilians', 'Explosions/Remote violence',
                         'Protests', 'Riots', 'Strategic developments']
        df_filtered = df_filtered[df_filtered['event_type'].isin(violence_types)]
    
    print(f"✓ Filtered data: {len(df_filtered)} events")
    print(f"  Countries: {df_filtered['country'].unique()}")
    
    return df_filtered

# Apply filtering if data loaded
if df_raw is not None:
    df_filtered = load_and_filter_acled(df_raw, COUNTRIES)
else:
    df_filtered = None

## Step 2: Monthly Aggregation

Aggregate daily event counts to monthly totals per country. This creates the time series data used for lagged analysis.

In [ ]:
def aggregate_to_monthly(df, countries):
    """Aggregate daily events to monthly counts"""
    
    if df is None:
        return None
    
    # Extract year-month
    df['year_month'] = df['date'].dt.to_period('M')
    
    # Count events by country and month
    monthly_counts = df.groupby(['year_month', 'country']).size().reset_index(name='events')
    monthly_counts['date'] = monthly_counts['year_month'].dt.to_timestamp()
    
    # Create complete date range for all countries
    date_range = pd.date_range(start='2020-01-01', end='2021-01-31', freq='MS')
    country_date_index = pd.MultiIndex.from_product([date_range, countries], names=['date', 'country'])
    monthly_full = pd.DataFrame(index=country_date_index).reset_index()
    
    # Merge with counts
    monthly_counts = monthly_counts[['date', 'country', 'events']]
    monthly_full = monthly_full.merge(monthly_counts, on=['date', 'country'], how='left')
    monthly_full['events'] = monthly_full['events'].fillna(0).astype(int)
    
    print(f"✓ Aggregated to {len(date_range)} months × {len(countries)} countries = {len(monthly_full)} records")
    
    return monthly_full

# Apply aggregation
if df_filtered is not None:
    df_monthly = aggregate_to_monthly(df_filtered, COUNTRIES)
    print("\nMonthly aggregated data (first 10 rows):")
    print(df_monthly.head(10))
else:
    df_monthly = None
    print("Skipping aggregation - no filtered data available")

## Step 3: Create Lagged Variables and Reshape

Create lagged structure:
- Afghanistan events at time t
- Neighbor events at time t+1 (one-month lag)

This lagged design allows us to test spillover effects temporally.

In [ ]:
def create_lagged_structure(df_monthly, neighbors):
    """Create lagged regression dataset"""
    
    if df_monthly is None:
        return None
    
    # Pivot to wide format
    df_wide = df_monthly.pivot(index='date', columns='country', values='events')
    df_wide = df_wide.fillna(0).astype(int)
    
    # Create lagged variables for neighbors (t+1)
    for neighbor in neighbors:
        df_wide[f'{neighbor}_events_tplus1'] = df_wide[neighbor].shift(-1)
    
    # Rename Afghanistan column
    df_wide = df_wide.rename(columns={'Afghanistan': 'Afghanistan_events'})
    
    # Remove last row (incomplete lag)
    df_wide = df_wide.iloc[:-1].reset_index()
    
    print(f"✓ Created lagged structure: {len(df_wide)} obs × {len(df_wide.columns)} variables")
    print(f"\n  Variables:")
    print(f"  - Afghanistan_events (t)")
    for neighbor in neighbors:
        print(f"  - {neighbor}_events_tplus1")
    
    return df_wide

# Create lagged structure
if df_monthly is not None:
    df_lagged = create_lagged_structure(df_monthly, NEIGHBORS)
    print("\nFinal dataset structure (first 5 rows):")
    print(df_lagged.head())
    print("\nData types:")
    print(df_lagged.dtypes)
else:
    df_lagged = None

## Save Processed Data

Store the cleaned dataset for use in EDA and analysis notebooks.

In [ ]:
# Save cleaned data
if df_lagged is not None:
    output_path = os.path.join(data_dir, 'processed_spillover_data.csv')
    df_lagged.to_csv(output_path, index=False)
    print(f"✓ Saved cleaned data to: {output_path}")
else:
    print("Warning: No data to save. Please load ACLED data first.")